In [29]:
import requests
import json
import re
import pandas as pd
import numpy as np

In [30]:
def get_uniprot(accession):
    
    endpoint = f"https://rest.uniprot.org/uniprotkb/{accession}.json"
    r = requests.get(endpoint)

    if r.status_code != 200:
        raise Exception("Invalid UniProt ID")

    return r.json()


def uniprot_parse_response(resp: dict):

    organism = resp.get("organism", {}).get("scientificName")

    geneInfo = None
    if resp.get("genes"):
        geneInfo = resp["genes"][0].get("geneName", {}).get("value")

    sequenceInfo = resp.get("sequence", {}).get("length")

    type_info = resp.get("entryType")

    output = {
        "organism": organism,
        "geneInfo": geneInfo,
        "sequenceInfo": sequenceInfo,
        "type": type_info
    }

    return output

In [31]:
def get_ensembl(id):
    
    endpoint = f"https://rest.ensembl.org/lookup/id/{id}?content-type=application/json"

    r = requests.get(endpoint)

    if r.status_code != 200:
        raise Exception("Invalid Ensembl ID")

    return r.json()


def ensembl_parse_response(resp: dict):

    output = {
        "object_type": resp.get("object_type"),
        "assembly_name": resp.get("assembly_name"),
        "species": resp.get("species"),
        "db_type": resp.get("db_type"),
        "biotype": resp.get("biotype"),
        "display_name": resp.get("display_name"),
        "id": resp.get("id"),
        "description": resp.get("description"),
        "canonical_transcript": resp.get("canonical_transcript"),
        "source": resp.get("source")
    }

    return output

In [32]:
def main(ids: list):

    results = {}

    for i in ids:

        try:

            if re.match(r"^ENS", i):
                resp = get_ensembl(i)
                parsed = ensembl_parse_response(resp)

            else:
                resp = get_uniprot(i)
                parsed = uniprot_parse_response(resp)

            results[i] = parsed

        except Exception as e:
            results[i] = {"error": str(e)}

    df = pd.DataFrame.from_dict(results, orient="index")

    return df

In [33]:
ids = ['P11473', 'Q91XI3', 'hello', 'ENSG00000157764', 'ENSG00000139618']

df = main(ids)
df

,organism,geneInfo,sequenceInfo,type,error,object_type,assembly_name,species,db_type,biotype,display_name,id,description,canonical_transcript,source
P11473,Homo sapiens,VDR,427.0,UniProtKB reviewed (Swiss-Prot),NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Q91XI3,Ictidomys tridecemlineatus,INS,110.0,UniProtKB reviewed (Swiss-Prot),NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
hello,NaN,NaN,NaN,NaN,Invalid UniProt ID,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
ENSG00000157764,NaN,NaN,NaN,NaN,NaN,Gene,GRCh38,homo_sapiens,core,protein_coding,BRAF,ENSG00000157764,"B-Raf proto-oncogene, serine/threonine kinase ...",ENST00000646891.2,ensembl_havana
ENSG00000139618,NaN,NaN,NaN,NaN,NaN,Gene,GRCh38,homo_sapiens,core,protein_coding,BRCA2,ENSG00000139618,BRCA2 DNA repair associated [Source:HGNC Symbo...,ENST00000380152.8,ensembl_havana
